# Module 20: Flash Sale Inventory Reservation Amazon — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/flash_sale_engine.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import flash_sale_engine

classes = [n for n, o in inspect.getmembers(flash_sale_engine, inspect.isclass)
           if o.__module__ == 'flash_sale_engine']
functions = [n for n, o in inspect.getmembers(flash_sale_engine, inspect.isfunction)
             if o.__module__ == 'flash_sale_engine']

print('module   : flash_sale_engine')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(flash_sale_engine, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Atomic reservation and overselling prevention

This is the module's own `test_atomic_reservation_and_overselling_prevention` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from flash_sale_engine import (
    AtomicInventoryReservationManager,
    ReservationStatus,
)

mgr = AtomicInventoryReservationManager()
sku = "PLAYSTATION_5"
mgr.register_sku(sku, total_stock=5)

# Reserve 3 units
ok1, tok1 = mgr.reserve(sku, "user_1", quantity=3, ttl_sec=60.0, current_time=0.0)
assert ok1 is True
assert tok1 is not None
assert tok1.quantity == 3
assert tok1.status == ReservationStatus.RESERVED

# Reserve 2 units
ok2, tok2 = mgr.reserve(sku, "user_2", quantity=2, ttl_sec=60.0, current_time=0.0)
assert ok2 is True

# Try reserving 1 more unit (available is 0)
ok3, tok3 = mgr.reserve(sku, "user_3", quantity=1, ttl_sec=60.0, current_time=0.0)
assert ok3 is False
assert tok3 is None

snap = mgr.get_stock_snapshot(sku)
assert snap["available"] == 0
assert snap["reserved"] == 5
assert snap["purchased"] == 0
assert mgr.verify_conservation_invariant(sku) is True

print('PASSED: test_atomic_reservation_and_overselling_prevention')

## 3. 🔮 Prediction — commit before you run

1,000 users race for 50 items. Predict how many succeed under a naive read-then-write, and how many under an atomic reservation.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_confirm_purchase_lifecycle`, which tests exactly this property.


In [ ]:
mgr = AtomicInventoryReservationManager()
sku = "TAYLOR_SWIFT_VIP"
mgr.register_sku(sku, total_stock=10)

ok, tok = mgr.reserve(sku, "fan_1", quantity=2, ttl_sec=100.0, current_time=10.0)
assert ok is True

# Confirm purchase before expiration (T=50s <= T=110s)
success = mgr.confirm_purchase(tok.reservation_id, current_time=50.0)
assert success is True
assert tok.status == ReservationStatus.PURCHASED

snap = mgr.get_stock_snapshot(sku)
assert snap["available"] == 8
assert snap["reserved"] == 0
assert snap["purchased"] == 2
assert mgr.verify_conservation_invariant(sku) is True

# Confirming already purchased reservation should return False
assert mgr.confirm_purchase(tok.reservation_id, current_time=55.0) is False

print('PASSED: test_confirm_purchase_lifecycle')

## 4. Measure it: Ttl expiry and reaper rollback

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_ttl_expiry_and_reaper_rollback` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

mgr = AtomicInventoryReservationManager()
sku = "LIMITED_EDITION_SNEAKER"
mgr.register_sku(sku, total_stock=4)

ok, tok = mgr.reserve(sku, "sneakerhead", quantity=3, ttl_sec=30.0, current_time=0.0)
assert ok is True
assert tok.expires_at == 30.0

# User attempts to confirm AFTER expiration at T=40.0s
confirmed = mgr.confirm_purchase(tok.reservation_id, current_time=40.0)
assert confirmed is False
assert tok.status == ReservationStatus.EXPIRED

# Stock should have been restored back to available
snap = mgr.get_stock_snapshot(sku)
assert snap["available"] == 4
assert snap["reserved"] == 0
assert mgr.verify_conservation_invariant(sku) is True

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_ttl_expiry_and_reaper_rollback')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(flash_sale_engine) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Read-then-write under contention oversells. Atomicity is the whole fix.
2. A reservation with a TTL converts a race into a queue.
3. Assert the conservation invariant: sold + reserved + available never changes.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
